# Problem 52: Conversational RAG

**Difficulty:** Medium | **Framework:** LangChain

**Categories:** rag, memory

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/conversational-rag).

## 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-openai langchain-community langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The agent must maintain conversation history across turns
- Follow-up questions must be resolved using prior context
- The retriever must receive contextualized queries, not raw follow-ups
- The agent must still retrieve documents for each turn


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings()

documents = [
    Document(page_content="ProductX is an enterprise CRM platform starting at $99/month."),
    Document(page_content="ProductY is a project management tool priced at $49/month per team."),
    Document(page_content="ProductX offers advanced analytics, custom dashboards, and API access."),
    Document(page_content="ProductY includes Gantt charts, sprint planning, and time tracking."),
    Document(page_content="ProductX has a 14-day free trial with full feature access."),
]

vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based on context.\n\nContext: {context}"),
    ("human", "{question}"),
])

# BUG: Each query is treated independently — no conversation history
def ask(question: str) -> str:
    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])
    chain = prompt | llm
    result = chain.invoke({"context": context, "question": question})
    return result.content

print(ask("Tell me about ProductX."))
# Follow-up: "What about pricing?" — but the agent doesn't know we're still talking about ProductX
print(ask("What about pricing?"))

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Store conversation history and use it to reformulate follow-up questions into standalone queries
2. Before retrieval, combine the chat history with the current question to create a contextualized search query
3. Use a history-aware retriever or a question condensation step before the main retrieval


## 7. Evaluation Criteria

Check your solution against these criteria:

- Maintains conversation history
- Resolves follow-up references
- Retrieves with conversational context
